In [1]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/50.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/50.1 MB 105.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 41.7/50.1 MB 111.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 112.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 112.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 112.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.1/50.1 MB 112.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 37.7 MB/s  0:00:01



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: /n/home10/jrochatrindade/.gntweets_env/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import gc
import glob
import gzip
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from multiprocessing import Pool, cpu_count

In [3]:
# -------------------------------------------------------------
# 1. Base Paths & Configuration
# -------------------------------------------------------------
RAW_DATA_DIR    = "/n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/"
SPATIAL_DIR     = "/n/netscratch/cga/Everyone/Gun_voilence/notebooks/spatial/"
TARGET_ROOT_DIR = "/n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day"

NUM_WORKERS = min(32, cpu_count() or 1)
CHUNK_SIZE  = 250_000
LOG_INTERVAL_BATCHES = 20
FLUSH_INTERVAL_FILES = 500

START_DATE = pd.to_datetime("2010-01-01 00:00:00")
END_DATE   = pd.to_datetime("2023-12-31 23:59:59")
TARGET_YEARS = [str(year) for year in range(2010, 2024)]

REGION_CONFIG = {
    'IL': {
        'name': 'israel',
        'iso_code': 'ISR',
        'shapefile': "adm1_simplify.shp",
        'iso_filter': ['ISR'],
        'plot_file': "validation_intersected_israel.png"
    },
    'HK': {
        'name': 'hong_kong',
        'iso_code': 'HKG',
        'shapefile': "adm1_simplify.shp",
        'iso_filter': ['HKG'],
        'plot_file': "validation_intersected_hong_kong.png"
    },
    'CH': {
        'name': 'china',
        'iso_code': 'CHN',
        'shapefile': "adm2_simplify.shp",
        'iso_filter': ['CHN'],
        'plot_file': "validation_intersected_china.png"
    }
}

EXPECTED_COLUMNS = [
    'message_id', 'date', 'text', 'tags', 'tweet_lang', 'source', 'place', 'geom',
    'retweets', 'tweet_favorites', 'photo_url', 'quoted_status_id', 'user_id',
    'user_name', 'user_location', 'followers', 'friends', 'user_favorites',
    'status', 'user_lang', 'latitude', 'longitude', 'data_source', 'GPS',
    'spatialerror', 'OBJECTID', 'ID_0', 'NAME_0', 'ISO', 'ID_1', 'NAME_1',
    'ID_2', 'NAME_2'
]

GEO_COLS = ["OBJECTID", "ID_0", "NAME_0", "ISO", "ID_1", "NAME_1", "ID_2", "NAME_2"]

In [4]:
# -------------------------------------------------------------
# 2. Worker Functions (Defined with consistent names)
# -------------------------------------------------------------
WORKER_POLY_GDFS = None
WORKER_BBOXES = None

def init_worker(poly_gdfs, bboxes):
    """Initializes polygon layers once per worker core."""
    global WORKER_POLY_GDFS, WORKER_BBOXES
    WORKER_POLY_GDFS = poly_gdfs
    WORKER_BBOXES = bboxes

def should_process_file(file_path):
    """Filters filenames to include only records from 2010 to 2023."""
    basename = os.path.basename(file_path)
    match = re.search(r"(\d{4})_(\d{2})_(\d{2})", basename)
    if match:
        year = int(match.group(1))
        return 2010 <= year <= 2023
    return True

def process_and_intersect_file_worker(file_path):
    """Ingests raw file, keeps all 24 metadata fields, performs BBox & spatial join."""
    global WORKER_POLY_GDFS, WORKER_BBOXES
    empty_result = {code: pd.DataFrame() for code in REGION_CONFIG}

    if not should_process_file(file_path):
        return empty_result

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8', errors='replace') as f:
            df = pd.read_csv(f, sep='\t', on_bad_lines="skip", dtype=str)

        # Standardize date column naming
        if 'date' not in df.columns and 'tweet_date' in df.columns:
            df['date'] = df['tweet_date']
        elif 'date' in df.columns and 'tweet_date' in df.columns:
            df['date'] = df['date'].fillna(df['tweet_date'])

        required_cols = ['message_id', 'date', 'latitude', 'longitude']
        if not all(c in df.columns for c in required_cols):
            return empty_result

        # Convert coordinates and date for filtering
        df['lat_num'] = pd.to_numeric(df['latitude'], errors='coerce')
        df['lon_num'] = pd.to_numeric(df['longitude'], errors='coerce')
        df['parsed_date'] = pd.to_datetime(df['date'], errors='coerce')

        df = df.dropna(subset=['message_id', 'parsed_date', 'lat_num', 'lon_num'])
        df = df[(df['parsed_date'] >= START_DATE) & (df['parsed_date'] <= END_DATE)]

        if df.empty:
            return empty_result

        results = {}

        for reg_code, poly_gdf in WORKER_POLY_GDFS.items():
            min_lon, min_lat, max_lon, max_lat = WORKER_BBOXES[reg_code]

            # 1. Vectorized Bounding Box filter
            bbox_mask = (
                (df['lat_num'] >= min_lat) & (df['lat_num'] <= max_lat) &
                (df['lon_num'] >= min_lon) & (df['lon_num'] <= max_lon)
            )
            candidates = df[bbox_mask].copy()

            if candidates.empty:
                results[reg_code] = pd.DataFrame()
                continue

            # 2. Point geometry generation
            points = [Point(xy) for xy in zip(candidates['lon_num'], candidates['lat_num'])]
            points_gdf = gpd.GeoDataFrame(candidates, geometry=points, crs="EPSG:4326")

            # 3. Inner Point-in-Polygon spatial join
            joined = gpd.sjoin(points_gdf, poly_gdf, how="inner", predicate="within")

            if not joined.empty:
                joined['day'] = joined['parsed_date'].dt.strftime('%Y-%m-%d')
                joined['geom'] = "POINT(" + joined['longitude'].astype(str) + " " + joined['latitude'].astype(str) + ")"

                for col in EXPECTED_COLUMNS:
                    if col not in joined.columns:
                        joined[col] = None

                for col in EXPECTED_COLUMNS:
                    joined[col] = joined[col].astype(str).replace({'nan': None, 'None': None, '<NA>': None})

                results[reg_code] = pd.DataFrame(joined[EXPECTED_COLUMNS + ['day']])
            else:
                results[reg_code] = pd.DataFrame()

        return results

    except Exception:
        return empty_result

In [5]:
# -------------------------------------------------------------
# 3. Load Shapefiles & Extract Bounding Boxes
# -------------------------------------------------------------
print(f"--> Step 1/3: Loading spatial boundaries ({NUM_WORKERS} cores configured)...", flush=True)

MASTER_POLY_GDFS = {}
MASTER_BBOXES    = {}

for reg_code, conf in REGION_CONFIG.items():
    iso = conf['iso_code']
    country_out_dir = os.path.join(TARGET_ROOT_DIR, iso)
    os.makedirs(country_out_dir, exist_ok=True)

    shp_path = os.path.join(SPATIAL_DIR, conf['shapefile'])
    gdf = gpd.read_file(shp_path)

    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    country_col = 'NAME_0' if 'NAME_0' in gdf.columns else ('NAME' if 'NAME' in gdf.columns else None)
    mask = gdf['ISO'].isin(conf['iso_filter'])
    if country_col:
        mask = mask | gdf[country_col].str.contains(conf['name'].replace('_', ' '), case=False, na=False)

    filtered_gdf = gdf[mask].copy()

    for c in GEO_COLS:
        if c not in filtered_gdf.columns:
            filtered_gdf[c] = None

    filtered_gdf = filtered_gdf[GEO_COLS + ['geometry']]

    minx, miny, maxx, maxy = filtered_gdf.total_bounds
    MASTER_POLY_GDFS[reg_code] = filtered_gdf
    MASTER_BBOXES[reg_code]    = (minx, miny, maxx, maxy)

    print(f"    • [{reg_code}] {conf['name'].upper()}: {len(filtered_gdf)} polygons loaded. BBox: {MASTER_BBOXES[reg_code]}", flush=True)

    del gdf
    gc.collect()

# Discover raw files
print("\nDiscovering raw archive files in repository...", flush=True)
all_files = []
for year in TARGET_YEARS:
    year_path = os.path.join(RAW_DATA_DIR, year, "**", "*.csv.gz")
    found = glob.glob(year_path, recursive=True)
    all_files.extend(found)

filtered_files = [f for f in all_files if should_process_file(f)]
total_files = len(filtered_files)
print(f"Total raw archive files queued for processing: {total_files:,}\n", flush=True)

--> Step 1/3: Loading spatial boundaries (32 cores configured)...


    • [IL] ISRAEL: 7 polygons loaded. BBox: (np.float64(34.2680092), np.float64(29.4970837), np.float64(35.9009361), np.float64(33.3531113))


    • [HK] HONG_KONG: 18 polygons loaded. BBox: (np.float64(113.8395844), np.float64(22.1559715), np.float64(114.4390259), np.float64(22.5620937))


    • [CH] CHINA: 326 polygons loaded. BBox: (np.float64(73.5577011), np.float64(18.1615295), np.float64(134.7739258), np.float64(53.5608597))



Discovering raw archive files in repository...


Total raw archive files queued for processing: 98,584



In [6]:
# -------------------------------------------------------------
# 4. Streamed Execution with Compact Progress Summary
# -------------------------------------------------------------
print("\n--> Step 2/3: Executing parallel spatial joins...", flush=True)

print("=" * 75, flush=True)
print(f"Processing Tweets for [IL, HK, CH] across {total_files:,} archives with {NUM_WORKERS} cores", flush=True)
print(f"Target Directory: {TARGET_ROOT_DIR}/<ISO>/YYYY-MM-DD.parquet", flush=True)
print("=" * 75, flush=True)

country_buffers = {conf['iso_code']: [] for conf in REGION_CONFIG.values()}
total_intersected_counts = {conf['iso_code']: 0 for conf in REGION_CONFIG.values()}
diagnostic_samples = {conf['iso_code']: [] for conf in REGION_CONFIG.values()}

total_processed_files = 0

def flush_country_buffers_to_parquet(buffers):
    """Merges in-memory records and appends to daily Parquet files."""
    for iso_code, dfs in buffers.items():
        if not dfs:
            continue
        merged_batch = pd.concat(dfs, ignore_index=True)
        for day_val, day_group in merged_batch.groupby('day'):
            target_parquet = os.path.join(TARGET_ROOT_DIR, iso_code, f"{day_val}.parquet")
            clean_group = day_group[EXPECTED_COLUMNS].copy()

            if os.path.exists(target_parquet):
                existing_df = pd.read_parquet(target_parquet)
                clean_group = pd.concat([existing_df, clean_group], ignore_index=True)

            clean_group = clean_group.drop_duplicates(subset=['message_id'], keep='last')

            clean_group.to_parquet(
                target_parquet,
                index=True,
                engine='pyarrow',
                compression='snappy'
            )
        buffers[iso_code] = []

with Pool(processes=NUM_WORKERS, initializer=init_worker, initargs=(MASTER_POLY_GDFS, MASTER_BBOXES)) as pool:
    for file_idx, result_dict in enumerate(pool.imap_unordered(process_and_intersect_file_worker, filtered_files, chunksize=10)):
        total_processed_files += 1

        for reg_code, df_matched in result_dict.items():
            if not df_matched.empty:
                iso = REGION_CONFIG[reg_code]['iso_code']
                matched_count = len(df_matched)
                country_buffers[iso].append(df_matched)
                total_intersected_counts[iso] += matched_count

                if len(diagnostic_samples[iso]) < 50_000:
                    sample_count = min(100, matched_count)
                    diagnostic_samples[iso].append(df_matched[['latitude', 'longitude']].sample(n=sample_count, random_state=42))

        if total_processed_files % FLUSH_INTERVAL_FILES == 0:
            flush_country_buffers_to_parquet(country_buffers)
            gc.collect()

        if total_processed_files % LOG_INTERVAL_BATCHES == 0:
            pct = min(100.0, (total_processed_files / total_files) * 100)
            print(
                f"   [{pct:5.1f}%] Files: {total_processed_files:,} / {total_files:,} | "
                f"Matched: IL={total_intersected_counts['ISR']:,} | "
                f"HK={total_intersected_counts['HKG']:,} | "
                f"CH={total_intersected_counts['CHN']:,}",
                flush=True
            )

# Final flush
flush_country_buffers_to_parquet(country_buffers)
del country_buffers
gc.collect()

print("\n" + "=" * 75, flush=True)
print(f"--> COMPLETED EXTRACTION & PARTITIONING:", flush=True)
for reg_code, conf in REGION_CONFIG.items():
    iso = conf['iso_code']
    print(f"    • [{reg_code}] {conf['name'].upper()} ({iso}): {total_intersected_counts[iso]:,} tweets saved.", flush=True)
print("=" * 75 + "\n", flush=True)


--> Step 2/3: Executing parallel spatial joins...


Processing Tweets for [IL, HK, CH] across 98,584 archives with 32 cores


Target Directory: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/<ISO>/YYYY-MM-DD.parquet


   [  0.0%] Files: 20 / 98,584 | Matched: IL=0 | HK=0 | CH=0


   [  0.0%] Files: 40 / 98,584 | Matched: IL=0 | HK=0 | CH=0


...

   [100.0%] Files: 98,580 / 98,584 | Matched: IL=16,089,323 | HK=5,240,531 | CH=16,148,154


--> COMPLETED EXTRACTION & PARTITIONING:


    • [IL] ISRAEL (ISR): 16,089,656 tweets saved.


    • [HK] HONG_KONG (HKG): 5,240,579 tweets saved.


    • [CH] CHINA (CHN): 16,148,499 tweets saved.


In [7]:
# -------------------------------------------------------------
# 5. Diagnostic Validation Plots
# -------------------------------------------------------------
print("--> Step 3/3: Generating validation plots...", flush=True)

for reg_code, conf in REGION_CONFIG.items():
    iso = conf['iso_code']
    poly_gdf = MASTER_POLY_GDFS[reg_code]
    samples_list = diagnostic_samples[iso]
    plot_file_path = os.path.join(TARGET_ROOT_DIR, iso, conf['plot_file'])

    if samples_list:
        sample_df = pd.concat(samples_list, ignore_index=True)
        if len(sample_df) > 50_000:
            sample_df = sample_df.sample(n=50_000, random_state=42)

        sample_pts = gpd.GeoDataFrame(
            sample_df,
            geometry=gpd.points_from_xy(sample_df.longitude.astype(float), sample_df.latitude.astype(float)),
            crs="EPSG:4326"
        )

        fig, ax = plt.subplots(figsize=(12, 10))
        poly_gdf.plot(ax=ax, color="#f0f0f0", edgecolor="#333333", linewidth=0.5, alpha=0.9)
        sample_pts.plot(ax=ax, color="#d73027", markersize=0.5, alpha=0.5, label=f"Intersected Sample (N={len(sample_df):,})")

        ax.set_title(f"Spatial Intersection Validation - {conf['name'].upper()} ({iso})", fontsize=14, pad=12)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.legend(loc="upper right", markerscale=8)
        ax.grid(True, linestyle="--", alpha=0.3)

        plt.savefig(plot_file_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"    • Validation plot saved to: {plot_file_path}", flush=True)

    del samples_list
    gc.collect()

print("\nAll regions processed, daily Parquet files written, and plots generated successfully!", flush=True)

--> Step 3/3: Generating validation plots...


    • Validation plot saved to: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/ISR/validation_intersected_israel.png


    • Validation plot saved to: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/HKG/validation_intersected_hong_kong.png


    • Validation plot saved to: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/CHN/validation_intersected_china.png



All regions processed, daily Parquet files written, and plots generated successfully!


In [8]:
import os
import glob
import pyarrow.parquet as pq
import pandas as pd

# -------------------------------------------------------------
# 6. Base Paths & Target Schema Definition
# -------------------------------------------------------------
TARGET_ROOT_DIR = "/n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day"

# The exact 33 data columns required by the pipeline
EXPECTED_COLUMNS = [
    'message_id', 'date', 'text', 'tags', 'tweet_lang', 'source', 'place', 'geom',
    'retweets', 'tweet_favorites', 'photo_url', 'quoted_status_id', 'user_id',
    'user_name', 'user_location', 'followers', 'friends', 'user_favorites',
    'status', 'user_lang', 'latitude', 'longitude', 'data_source', 'GPS',
    'spatialerror', 'OBJECTID', 'ID_0', 'NAME_0', 'ISO', 'ID_1', 'NAME_1',
    'ID_2', 'NAME_2'
]

# Columns that are expected to contain legitimate null values
NULL_CANDIDATE_COLUMNS = ['tweet_lang', 'place', 'tweet_favorites', 'photo_url']

print("=" * 85)
print("PARQUET DATA INTEGRITY, SCHEMA, AND NULL-VALUE AUDIT")
print("=" * 85)

for iso_code in ['ISR', 'HKG', 'CHN']:
    country_folder = os.path.join(TARGET_ROOT_DIR, iso_code)
    parquet_files = sorted(glob.glob(os.path.join(country_folder, "*.parquet")))

    print(f"\n--> Region [{iso_code}]: {country_folder}")

    if not parquet_files:
        print("    [WARNING] No Parquet files found in this folder yet. Skipping.")
        continue

    sample_file = parquet_files[0]
    print(f"    • Total daily Parquet files found: {len(parquet_files):,}")
    print(f"    • Inspecting sample file: {os.path.basename(sample_file)}")

    # ---------------------------------------------------------
    # 6.1 PyArrow Physical Metadata & Schema Inspection
    # ---------------------------------------------------------
    parquet_metadata = pq.read_metadata(sample_file)
    parquet_schema = pq.read_schema(sample_file)
    actual_columns = parquet_schema.names

    print("\n    [1. SCHEMA AND COLUMN VERIFICATION]")
    print(f"    • Record count in sample: {parquet_metadata.num_rows:,} rows")
    print(f"    • Physical columns in file: {len(actual_columns)} (Expected: 34 = 33 data cols + 1 index)")

    has_index = '__index_level_0__' in actual_columns
    print(f"    • Pandas row index (__index_level_0__) present: {'PASS' if has_index else 'FAIL'}")

    data_columns = [c for c in actual_columns if c != '__index_level_0__']
    missing_cols = set(EXPECTED_COLUMNS) - set(data_columns)
    extra_cols = set(data_columns) - set(EXPECTED_COLUMNS)

    if not missing_cols and not extra_cols:
        print("    • Column alignment: 100% MATCH with expected 33-column schema.")
    else:
        if missing_cols:
            print(f"    • [ERROR] Missing columns: {sorted(missing_cols)}")
        if extra_cols:
            print(f"    • [WARNING] Extra unexpected columns: {sorted(extra_cols)}")

    # ---------------------------------------------------------
    # 6.2 Native NULL vs. Literal String ('nan'/'None') Audit
    # ---------------------------------------------------------
    df_sample = pd.read_parquet(sample_file)

    print("\n    [2. NULL-VALUE INTEGRITY AUDIT]")
    for col in NULL_CANDIDATE_COLUMNS:
        if col in df_sample.columns:
            has_str_nan = (df_sample[col] == 'nan').any()
            has_str_none = (df_sample[col] == 'None').any()
            null_count = df_sample[col].isna().sum()
            total_rows = len(df_sample)

            if has_str_nan or has_str_none:
                print(f"    • Column '{col}': [FAIL] Contains literal string 'nan' or 'None'!")
            else:
                print(f"    • Column '{col}': [PASS] Stored as true native NULL ({null_count:,}/{total_rows:,} empty)")

    # ---------------------------------------------------------
    # 6.3 Data Preview
    # ---------------------------------------------------------
    print("\n    [3. SAMPLE RECORDS PREVIEW]")
    preview_cols = ['message_id', 'date', 'geom', 'ISO', 'NAME_1', 'NAME_2']
    valid_preview_cols = [c for c in preview_cols if c in df_sample.columns]
    print(df_sample[valid_preview_cols].head(3).to_string(index=True))
    print("-" * 85)

print("\nValidation complete for all available regions.")

PARQUET DATA INTEGRITY, SCHEMA, AND NULL-VALUE AUDIT

--> Region [ISR]: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/ISR
    • Total daily Parquet files found: 3,965
    • Inspecting sample file: 2011-08-29.parquet

    [1. SCHEMA AND COLUMN VERIFICATION]
    • Record count in sample: 1 rows
    • Physical columns in file: 34 (Expected: 34 = 33 data cols + 1 index)
    • Pandas row index (__index_level_0__) present: PASS
    • Column alignment: 100% MATCH with expected 33-column schema.

    [2. NULL-VALUE INTEGRITY AUDIT]
    • Column 'tweet_lang': [PASS] Stored as true native NULL (1/1 empty)
    • Column 'place': [PASS] Stored as true native NULL (1/1 empty)
    • Column 'tweet_favorites': [PASS] Stored as true native NULL (1/1 empty)
    • Column 'photo_url': [PASS] Stored as true native NULL (1/1 empty)

    [3. SAMPLE RECORDS PREVIEW]
           message_id                 date                            geom  ISO    NAME_1 NAME_2
0  1081556709179

    • Column 'tweet_favorites': [PASS] Stored as true native NULL (20/20 empty)
    • Column 'photo_url': [PASS] Stored as true native NULL (19/20 empty)

    [3. SAMPLE RECORDS PREVIEW]
            message_id                 date                             geom  ISO        NAME_1 NAME_2
58  153689260057296896  2012-01-02 04:09:09  POINT(114.14036169 22.33962266)  HKG  Sham Shui Po    NaN
59  153725259772731392  2012-01-02 06:32:12  POINT(114.13622084 22.49123301)  HKG         North    NaN
60  153741525266870272  2012-01-02 07:36:50  POINT(114.20261264 22.34031137)  HKG  Wong Tai Sin    NaN
-------------------------------------------------------------------------------------

--> Region [CHN]: /n/netscratch/cga/Everyone/Gun_voilence/data/tweets_reorgnize_by_admin2_by_day/CHN
    • Total daily Parquet files found: 3,966
    • Inspecting sample file: 2010-11-13.parquet

    [1. SCHEMA AND COLUMN VERIFICATION]
    • Record count in sample: 1 rows
    • Physical columns in file: 34 (Expec